# Pancreas — Reliability / Uncertainty Evaluation (no retraining)

Accuracy is saturated (Fresnel/SEA don't beat baseline on Dice). This notebook pivots to the
**reliability axis**: it reuses the **existing per-seed checkpoints** and asks *which uncertainty
signal best predicts where the segmentation is wrong* — the quantity that matters clinically.

**Methods compared** (all on the same fixed held-out test set):
- `entropy` — predictive entropy of one model (1 forward pass; trivial baseline)
- `tta` — disagreement under test-time augmentations of one model (K passes, 1 model)
- `ensemble` — disagreement across the 5 seed models (**gold standard, 5× cost**)
- `fresnel_coh` — **multi-distance Fresnel coherence** of one model (**1 forward pass**): variance
  across the n_dist diffraction-propagated feature views. Talbot intuition: regions that *self-image
  stably* across propagation distances are confident; regions that *decohere* are uncertain.

**Metrics** (higher uncertainty should mark errors):
- **Failure-detection AUROC** — does the uncertainty map separate wrong from right voxels?
- **ECE** — is confidence calibrated to accuracy?
- **Risk–coverage AUC** — if the model abstains on its most uncertain voxels, how fast does error drop?

**The result that makes a paper:** if `fresnel_coh` (1 pass) matches `ensemble` (5 passes) on AUROC,
you get **ensemble-quality reliability at 1/5 the cost, grounded in optics** — a UVIF-aligned
contribution on the axis where accuracy was dead. No training; reuses your checkpoints.

## 1. Config — point at your cache + checkpoints

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
from pathlib import Path
import numpy as np, torch
DRIVE=Path("/content/drive/MyDrive")

class C:
    cache_root = DRIVE/"Datasets"/"PancreasCache"      # same cache used for training
    ckpt_dir   = DRIVE/"Outputs"/"Pancreas-Compare"/"Checkpoints"
    out_dir    = DRIVE/"Outputs"/"Pancreas-Reliability"
    seeds      = (0,1,2,3,4)
    test_seed  = 12345           # MUST match the training run (same fixed 38/42-case test set)
    test_fraction = 0.15
    target_shape = (112,112,112)
    base_channels = 16
    n_dist = 3                   # Fresnel propagation distances (match training)
    assess_arch = "fresnel"      # whose predictions we assess: "fresnel" or "baseline"
    tta_K = 6
    device = "cuda" if torch.cuda.is_available() else "cpu"
cfg=C()
# optional: copy cache to local disk to dodge Drive read flakiness
import shutil
loc=Path("/content/PancreasCache")
if not loc.exists() and cfg.cache_root.exists():
    shutil.copytree(cfg.cache_root, loc); cfg.cache_root=loc; print("cache -> local disk")
elif loc.exists(): cfg.cache_root=loc
OUT=cfg.out_dir; (OUT/"Tables").mkdir(parents=True,exist_ok=True); (OUT/"Figures").mkdir(parents=True,exist_ok=True)
print("device:",cfg.device,"| ckpts:",cfg.ckpt_dir)


## 2. Rebuild the SAME fixed test set + cache loader

In [ ]:
cache_files=sorted(p for p in cfg.cache_root.glob("*.npz"))
allidx=np.arange(len(cache_files)); np.random.default_rng(cfg.test_seed).shuffle(allidx)
ntest=int(len(cache_files)*cfg.test_fraction)
test_files=[cache_files[i] for i in allidx[:ntest]]
def detect(files,sample=60):
    vals=set()
    for p in files[:sample]: vals.update(int(v) for v in np.unique(np.load(p)["lab"]) if v>0)
    fg=sorted(vals); return {v:i+1 for i,v in enumerate(fg)}, len(fg)
LMAP,NFG=detect(cache_files); NCLS=NFG+1
CLASS_NAMES={1:"pancreas",2:"mass"} if NFG==2 else {c:f"class{c}" for c in range(1,NCLS)}
def load_case(p):
    d=np.load(p); img=d["img"].astype(np.float32); raw=d["lab"].astype(np.int64)
    img=(img-img.mean())/(img.std()+1e-6)
    lab=np.zeros_like(raw); 
    for v,k in LMAP.items(): lab[raw==v]=k
    return img.astype(np.float32), lab
print(f"{NCLS}-way {CLASS_NAMES} | fixed test cases: {len(test_files)}")


## 3. Models (match training) — Fresnel captures its per-distance views

In [ ]:
import math, torch.nn as nn, torch.nn.functional as F

class MultiBandSpectralEdgeAttention3D(nn.Module):
    def __init__(s,ch,nb=4):
        super().__init__(); s.mu=nn.Parameter(torch.linspace(0.45,0.9,nb)); s.log_sigma=nn.Parameter(torch.log(torch.full((nb,),0.08)))
        s.fuse=nn.Conv3d(ch*nb,ch,1); s.gate=nn.Sequential(nn.Conv3d(ch,ch,1),nn.Sigmoid()); s.n=nb
    def _rho(s,D,H,W,dev):
        fz=torch.fft.fftfreq(D,device=dev);fy=torch.fft.fftfreq(H,device=dev);fx=torch.fft.rfftfreq(W,device=dev)
        Z,Y,X=torch.meshgrid(fz,fy,fx,indexing="ij"); r=torch.sqrt(Z**2+Y**2+X**2); return r/(r.max()+1e-6)
    def forward(s,x):
        D,H,W=x.shape[-3:];Xf=torch.fft.rfftn(x.float(),dim=(-3,-2,-1));rho=s._rho(D,H,W,x.device)
        sg=torch.exp(s.log_sigma);mu=torch.clamp(s.mu,0,1)
        b=[torch.fft.irfftn(Xf*torch.exp(-((rho-mu[i])**2)/(2*sg[i]**2+1e-6)).to(Xf.dtype),s=(D,H,W),dim=(-3,-2,-1)).to(x.dtype) for i in range(s.n)]
        return x*(1+s.gate(s.fuse(torch.cat(b,1))))

class FresnelPropagation3D(nn.Module):
    def __init__(s,ch,nd=3):
        super().__init__(); s.z=nn.Parameter(torch.linspace(0.05,0.30,nd)); s.fuse=nn.Conv3d(ch*nd,ch,1); s.n=nd; s.capture=False; s.last=None
    def _k2(s,D,H,W,dev):
        fz=torch.fft.fftfreq(D,device=dev);fy=torch.fft.fftfreq(H,device=dev);fx=torch.fft.rfftfreq(W,device=dev)
        Z,Y,X=torch.meshgrid(fz,fy,fx,indexing="ij"); return Z**2+Y**2+X**2
    def forward(s,x):
        D,H,W=x.shape[-3:];Xf=torch.fft.rfftn(x.float(),dim=(-3,-2,-1));k2=s._k2(D,H,W,x.device)
        outs=[torch.fft.irfftn(Xf*torch.exp(1j*(2*math.pi)*s.z[i]*k2).to(Xf.dtype),s=(D,H,W),dim=(-3,-2,-1)).to(x.dtype) for i in range(s.n)]
        if s.capture: s.last=torch.stack(outs,0).detach()   # (n_dist, B, C, D,H,W)
        return x+s.fuse(torch.cat(outs,1))

class ResBlock3D(nn.Module):
    def __init__(s,ci,co):
        super().__init__();s.c1=nn.Conv3d(ci,co,3,padding=1);s.n1=nn.InstanceNorm3d(co);s.c2=nn.Conv3d(co,co,3,padding=1);s.n2=nn.InstanceNorm3d(co);s.skip=nn.Conv3d(ci,co,1) if ci!=co else nn.Identity()
    def forward(s,x):h=torch.relu(s.n1(s.c1(x)));h=s.n2(s.c2(h));return torch.relu(h+s.skip(x))
def make_module(a,ch,cfg):
    if a=="sea": return MultiBandSpectralEdgeAttention3D(ch,4)
    if a=="fresnel": return FresnelPropagation3D(ch,cfg.n_dist)
    return nn.Identity()
class CompareUNet3D(nn.Module):
    def __init__(s,cfg,n_classes,arch="baseline",in_ch=1):
        super().__init__();base=cfg.base_channels
        s.e1=ResBlock3D(in_ch,base);s.e2=ResBlock3D(base,2*base);s.bott=ResBlock3D(2*base,4*base)
        s.up2=nn.ConvTranspose3d(4*base,2*base,2,2);s.d2=ResBlock3D(4*base,2*base)
        s.up1=nn.ConvTranspose3d(2*base,base,2,2);s.d1=ResBlock3D(2*base,base)
        s.m2=make_module(arch,2*base,cfg);s.m1=make_module(arch,base,cfg)
        s.out=nn.Conv3d(base,n_classes,1);s.pool=nn.MaxPool3d(2)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.pool(e1));b=s.bott(s.pool(e2))
        d2=s.m2(s.d2(torch.cat([s.up2(b),e2],1)));d1=s.m1(s.d1(torch.cat([s.up1(d2),e1],1)))
        return s.out(d1)
def build(arch):
    return CompareUNet3D(cfg,NCLS,arch).to(cfg.device)
def load_seed(arch,seed):
    net=build(arch); p=cfg.ckpt_dir/f"{arch}_seed{seed}.pt"
    net.load_state_dict(torch.load(p,map_location=cfg.device)); net.eval(); return net
print("model defs ready")


## 4. Uncertainty estimators + metrics

In [ ]:
from scipy.ndimage import binary_dilation

def softmax_np(logits): 
    e=np.exp(logits-logits.max(0,keepdims=True)); return e/e.sum(0,keepdims=True)

@torch.no_grad()
def logits_of(net,img):
    x=torch.from_numpy(img)[None,None].to(cfg.device)
    return net(x)[0].cpu().numpy()            # (C,D,H,W)

@torch.no_grad()
def fresnel_coherence(net,img):
    # one forward pass; variance across n_dist propagated views at the final Fresnel layer
    for m in net.modules():
        if isinstance(m,FresnelPropagation3D): m.capture=True; m.last=None
    x=torch.from_numpy(img)[None,None].to(cfg.device); lo=net(x)[0].cpu().numpy()
    caps=[m.last for m in net.modules() if isinstance(m,FresnelPropagation3D) and m.last is not None]
    m_last=caps[-1]                            # (n_dist,B,C,d,h,w) at full-ish res (base channels)
    v=m_last.float().var(0).mean(1)[0]         # var across distances, mean over channels -> (d,h,w)
    u=torch.from_numpy(v.cpu().numpy()) if hasattr(v,'cpu') else v
    u=v.cpu().numpy()
    # upsample to full res
    u=torch.from_numpy(u)[None,None].float()
    u=F.interpolate(u,size=img.shape,mode="trilinear",align_corners=False)[0,0].numpy()
    for m in net.modules():
        if isinstance(m,FresnelPropagation3D): m.capture=False; m.last=None
    return lo,u

@torch.no_grad()
def tta_uncertainty(net,img,K=6):
    sm=[]; 
    base=softmax_np(logits_of(net,img)); sm.append(base)
    for k in range(K-1):
        aug=img+np.random.normal(0,0.03,img.shape).astype(np.float32)
        flips=[ax for ax in (0,1,2) if (k>>ax)&1]
        a=aug.copy()
        for ax in flips: a=np.flip(a,ax)
        s=softmax_np(logits_of(net,np.ascontiguousarray(a)))
        for ax in flips: s=np.flip(s,ax+1)
        sm.append(np.ascontiguousarray(s))
    sm=np.stack(sm,0)                          # (K,C,D,H,W)
    pbar=sm.mean(0); ent=-(pbar*np.log(pbar+1e-9)).sum(0)   # predictive entropy
    return pbar, ent

def entropy_u(prob): return -(prob*np.log(prob+1e-9)).sum(0)

# ---- metrics on an evaluation region R (band around structure) ----
def eval_region(pred,gt):
    fg=(pred>0)|(gt>0)
    return binary_dilation(fg,iterations=3)

def auroc(u,err):
    # Mann-Whitney AUROC: P(u_err > u_correct)
    pos=u[err==1]; neg=u[err==0]
    if len(pos)==0 or len(neg)==0: return np.nan
    # rank-based
    allv=np.concatenate([pos,neg]); order=allv.argsort(kind="mergesort"); ranks=np.empty(len(allv)); ranks[order]=np.arange(1,len(allv)+1)
    # average ties
    # (approx: ignore ties for speed on large arrays via argsort ranks)
    rp=ranks[:len(pos)].sum(); 
    return (rp-len(pos)*(len(pos)+1)/2)/(len(pos)*len(neg))

def ece(conf,correct,bins=10):
    e=0.0; n=len(conf)
    for i in range(bins):
        lo,hi=i/bins,(i+1)/bins; m=(conf>lo)&(conf<=hi)
        if m.sum()==0: continue
        e+=abs(correct[m].mean()-conf[m].mean())*m.sum()/n
    return e

def risk_coverage_auc(conf,correct,steps=20):
    order=np.argsort(-conf)  # most confident first
    c=correct[order]; aucs=[]
    for q in np.linspace(0.1,1.0,steps):
        k=max(1,int(len(c)*q)); aucs.append(1-c[:k].mean())   # risk = error rate at coverage q
    return float(np.mean(aucs))
print("estimators + metrics ready")


## 5. Run all estimators on the test set

In [ ]:
import pandas as pd
fres=[load_seed("fresnel",s) for s in cfg.seeds]
base=[load_seed("baseline",s) for s in cfg.seeds]
assess = fres if cfg.assess_arch=="fresnel" else base
m0=assess[0]

rows=[]
for fi,p in enumerate(test_files):
    img,gt=load_case(p); R=None
    # predictions of the single assessed model
    lo0=logits_of(m0,img); prob0=softmax_np(lo0); pred0=prob0.argmax(0)
    R=eval_region(pred0,gt); err=(pred0!=gt).astype(np.int8)
    conf0=prob0.max(0)
    # methods
    U={}
    U["entropy"]=entropy_u(prob0)
    _,U["tta"]=tta_uncertainty(m0,img,cfg.tta_K)
    if cfg.assess_arch=="fresnel":
        _,U["fresnel_coh"]=fresnel_coherence(m0,img)
    # ensemble (5 models): mean prob, entropy as uncertainty; ensemble has its OWN pred
    probs=[softmax_np(logits_of(net,img)) for net in assess]
    pbar=np.mean(probs,0); pred_e=pbar.argmax(0); err_e=(pred_e!=gt).astype(np.int8); conf_e=pbar.max(0)
    Ue=entropy_u(pbar)
    r=R.astype(bool)
    for name,u in U.items():
        rows.append(dict(case=p.stem,method=name,auroc=auroc(u[r],err[r]),
                         ece=ece(conf0[r],(pred0[r]==gt[r]).astype(float)),
                         rcauc=risk_coverage_auc(conf0[r],(pred0[r]==gt[r]).astype(float))))
    rows.append(dict(case=p.stem,method="ensemble",auroc=auroc(Ue[r],err_e[r]),
                     ece=ece(conf_e[r],(pred_e[r]==gt[r]).astype(float)),
                     rcauc=risk_coverage_auc(conf_e[r],(pred_e[r]==gt[r]).astype(float))))
    if (fi+1)%10==0: print(f"  {fi+1}/{len(test_files)} cases")
df=pd.DataFrame(rows)
summary=df.groupby("method").agg(auroc=("auroc","mean"),auroc_sd=("auroc","std"),
        ece=("ece","mean"),risk_cov_auc=("rcauc","mean")).round(4).reset_index().sort_values("auroc",ascending=False)
print("\n=== Failure-detection AUROC (higher=better) | ECE (lower=better) | Risk-Cov AUC (lower=better) ===")
print(summary.to_string(index=False))


## 6. Figures + save

In [ ]:
import matplotlib.pyplot as plt
df.to_csv(OUT/"Tables"/"reliability_per_case.csv",index=False)
summary.to_csv(OUT/"Tables"/"reliability_summary.csv",index=False)
fig,ax=plt.subplots(figsize=(7,4))
s=summary.sort_values("auroc")
ax.barh(s.method,s.auroc,xerr=s.auroc_sd,capsize=3,color="#4477aa")
ax.axvline(0.5,color="k",ls="--",lw=0.8); ax.set_xlabel("failure-detection AUROC"); ax.set_title("Which uncertainty predicts errors?")
plt.tight_layout(); fig.savefig(OUT/"Figures"/"auroc.png",dpi=160,bbox_inches="tight"); plt.show()

lines=["PANCREAS — RELIABILITY (uncertainty quality, no retraining)","="*60,
       f"assessed model: {cfg.assess_arch} | test cases: {len(test_files)} | seeds: {cfg.seeds}",""]
for _,r in summary.iterrows():
    lines.append(f"  {r.method:<12} AUROC={r.auroc:.4f}±{r.auroc_sd:.4f}  ECE={r.ece:.4f}  RiskCovAUC={r.risk_cov_auc:.4f}")
ens=summary[summary.method=='ensemble'].auroc.values[0]
if 'fresnel_coh' in summary.method.values:
    fc=summary[summary.method=='fresnel_coh'].auroc.values[0]
    lines+=["",f"fresnel_coh (1 pass) vs ensemble (5 passes): AUROC {fc:.4f} vs {ens:.4f}  (Δ={fc-ens:+.4f})",
            "If fresnel_coh ~ ensemble -> ensemble-quality reliability at 1/5 cost, grounded in optics."]
open(OUT/"reliability_summary.txt","w").write("\n".join(lines)+"\n")
print("\n".join(lines)); print("\nsaved ->",OUT)
